# VoxCPM2 模型在 魔搭社区 ModelScope GPU 上生成音频测试

## 关键修正（对照 Kaggle V5）
- ✅ ModelScope 免费 GPU 实例（单卡 V100/A100，按平台当日配额）
- ✅ 模型下载优先用 ModelScope SDK（魔搭内网最快），HF 镜像 requests 兜底
- ✅ 加载官方 `voxcpm` 库 `VoxCPM.from_pretrained(本地路径)`（失败回退项目源码）
- ✅ 不硬钉 torch，让 pip 拉 ≥2.5.0 兼容版
- ✅ 生成后写 .wav + RTF 测算
- ✅ 可一键转入算力池 Worker 模式（队列监听）

参考节点: kaggle-01 / paddle-01，本机复用同一 Redis 队列与 R2 存储。

## ⚠️ 红线#5 合规
以下 Secrets 均为占位符。请在 Notebook cell 中用 `os.environ` 注入
对应控制台轮换后的真值，请勿回填明文到本仓库。

In [ ]:
# === 在任何 import 之前设 HF 镜像 endpoint (huggingface_hub 导入时固化) ===
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_SSL_VERIFY"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

# ⚠️ 占位符：在执行下面 install + load 前用真实值替换 <...>
SECRETS = {
    "REDIS_HOST": "casual-sawfish-86152.upstash.io",
    "REDIS_PORT": "6379",
    "REDIS_AUTH": "<REDACTED_UPSTASH_REDIS_PASSWORD>",
    "R2_ENDPOINT": "https://<REDACTED_R2_ACCOUNT_ID>.r2.cloudflarestorage.com",
    "R2_ACCESS_KEY_ID": "<REDACTED_R2_ACCESS_KEY_ID>",
    "R2_SECRET_ACCESS_KEY": "<REDACTED_R2_SECRET_ACCESS_KEY>",
    "R2_BUCKET": "audiobook-assets",
    "R2_PUBLIC_URL": "https://pub-xxx.r2.dev",
    "WORKER_ID": "modelscope-v100-01",
    "VOXCPM2_MS_REPO": "OpenBMB/VoxCPM2,openbmb/VoxCPM2",
    "MODEL_CACHE": "/mnt/workspace/VoxCPM2",
    "VOXCPM2_PIP": "voxcpm==2.0.3",
}
for k, v in SECRETS.items():
    if not v.startswith("<"):
        os.environ.setdefault(k, v)
print("Secrets 注入完成（占位值已跳过，请确保上述 <...> 已替换为真值）")

In [ ]:
# 验证 GPU 环境（魔搭 Notebook 自带 torch）
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")

In [ ]:
# 安装官方推理库 voxcpm (README: pip install voxcpm) + 以防万一的依赖
import subprocess, sys
for pkg in ["voxcpm==2.0.3", "modelscope", "huggingface_hub", "requests",
            "soundfile", "boto3", "redis", "numpy"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", pkg], check=False)
print("Dependencies installed.")
import torch as _t
print(f"torch={_t.__version__}, cuda={_t.cuda.is_available()}")

In [ ]:
# 下载 VoxCPM2 模型 —— ModelScope SDK 优先，HF 镜像 requests 逐文件兜底
import os, sys, requests, json
from pathlib import Path

MODEL_DIR = Path(os.environ.get("MODEL_CACHE", "/mnt/workspace/VoxCPM2"))
MODEL_DIR.mkdir(parents=True, exist_ok=True)
HF_REPO = os.environ.get("VOXCPM2_HF_REPO", "openbmb/VoxCPM2")
ENDPOINT = os.environ.get("HF_ENDPOINT", "https://hf-mirror.com")
MS_CANDIDATES = [c.strip() for c in
                 os.environ.get("VOXCPM2_MS_REPO", "OpenBMB/VoxCPM2,openbmb/VoxCPM2").split(",")
                 if c.strip()]

# --- 方案 A: ModelScope SDK（魔搭平台内网最快）---
def try_modelscope():
    try:
        from modelscope import snapshot_download as ms
    except Exception as e:
        print(f"ModelScope SDK 不可用: {e}"); return False
    for repo in MS_CANDIDATES:
        try:
            print(f"📥 [ModelScope] {repo} -> {MODEL_DIR}")
            p = ms(repo, local_dir=str(MODEL_DIR))
            print(f"✅ 下载成功: {p}")
            if (MODEL_DIR / "config.json").exists():
                return True
        except Exception as e:
            print(f"⚠️ {repo} 失败: {str(e)[:100]}")
    return False

# --- 方案 B: HF 镜像 requests 逐文件（绕过 huggingface_hub HEAD bug）---
def try_hf_mirror():
    try:
        from huggingface_hub import HfApi
        api = HfApi(endpoint=ENDPOINT)
        info = api.model_info(HF_REPO)
        files = sorted(s.rfilename for s in info.siblings)
    except Exception as e:
        print(f"❌ HF 文件列表失败: {e}"); return False
    print(f"📂 {HF_REPO} 共 {len(files)} 文件 (base={ENDPOINT})")
    ok = 0
    for fname in files:
        dest = MODEL_DIR / fname
        if dest.exists() and dest.stat().st_size > 0:
            ok += 1; continue
        url = f"{ENDPOINT}/{HF_REPO}/resolve/main/{fname}"
        dest.parent.mkdir(parents=True, exist_ok=True)
        try:
            r = requests.get(url, timeout=180, stream=True, allow_redirects=True, verify=False)
            if r.status_code == 200:
                with open(dest, "wb") as f:
                    for c in r.iter_content(8192): f.write(c)
                ok += 1
                print(f"   ✅ {fname} ({dest.stat().st_size/1e6:.2f} MB)")
            else:
                print(f"   ❌ {fname}: HTTP {r.status_code}")
        except Exception as e:
            print(f"   ❌ {fname}: {type(e).__name__}: {str(e)[:80]}")
    return ok == len(files) and ok > 0

if (MODEL_DIR / "config.json").exists() and any(MODEL_DIR.glob("*.safetensors")):
    print(f"✅ 模型已缓存: {MODEL_DIR}")
elif try_modelscope() or try_hf_mirror():
    print("\n下载完成 ✅")
else:
    print("\n❌ 所有下载方案均失败")
print("目录内容:", os.listdir(MODEL_DIR))

In [ ]:
# 检查模型结构
import os
model_path = str(__import__("os").environ.get("MODEL_CACHE","/mnt/workspace/VoxCPM2"))
for root, dirs, files in os.walk(model_path):
    depth = root.replace(model_path, '').count(os.sep)
    if depth <= 2:
        for f in files:
            print(os.path.join(root, f))

In [ ]:
# 修复 config.json（补 model_type / dtype，与其它节点脚本一致）
import json
from pathlib import Path
cfg_path = Path(__import__("os").environ.get("MODEL_CACHE","/mnt/workspace/VoxCPM2")) / "config.json"
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text())
    if "model_type" not in cfg: cfg["model_type"] = "voxcpm2"
    if cfg.get("dtype") not in ("float16","fp16"): cfg["dtype"] = "float16"
    cfg_path.write_text(json.dumps(cfg, indent=2))
    print("✅ config.json 已修复:", json.dumps({k: cfg.get(k) for k in ["model_type","dtype"]}))
else:
    print(f"⚠️ 未找到 {cfg_path}")

In [ ]:
# 用官方 voxcpm 库加载 VoxCPM2 (本地已下载)
import time, sys, os
from pathlib import Path

model_path = str(Path(os.environ.get("MODEL_CACHE","/mnt/workspace/VoxCPM2")))
print(f"Loading VoxCPM2 from local: {model_path}")
t0 = time.time()
try:
    from voxcpm import VoxCPM
    model = VoxCPM.from_pretrained(
        model_path, load_denoiser=False, optimize=False, device="cuda",
    )
    sr = getattr(model.tts_model, "sample_rate", 24000)
    print(f"✅ 官方库加载完成, 耗时 {time.time()-t0:.1f}s, sr={sr}")
except Exception as e:
    print(f"官方库失败 ({e}), 回退项目源码 ...")
    sys.path.insert(0, "/mnt/workspace/src")  # 若已克隆仓库
    from voxcpm.model.voxcpm2 import VoxCPM2Model
    model = VoxCPM2Model.from_local(model_path, optimize=False)
    model.eval(); sr = 24000
    print(f"✅ 源码加载完成, 耗时 {time.time()-t0:.1f}s, sr={sr}")

In [ ]:
# VoxCPM2 推理测试 (官方 generate API) + 统一签名兼容
import time, os, numpy as np, soundfile as sf

texts = [
    "Hello, this is a test of VoxCPM2 text to speech synthesis.",
    "你好，这是 VoxCPM2 模型生成的中文语音测试。",
    "The quick brown fox jumps over the lazy dog.",
]

print(f"Starting generation (sr={sr})...")
total_rtf = 0.0
for i, text in enumerate(texts):
    print(f"\nTest {i+1}: {text[:50]}")
    try:
        t0 = time.time()
        try:
            wav = model.generate(text=text, cfg_value=2.0, inference_timesteps=10)
        except TypeError:
            wav = model.generate(target_text=text, max_len=1024, cfg_value=2.0, inference_timesteps=10)
        t1 = time.time()
        synth = t1 - t0
        wav = np.asarray(wav).astype(np.float32).reshape(-1)
        dur = len(wav) / sr
        rtf = synth / dur if dur > 0 else 0
        total_rtf += rtf
        out = f"/mnt/workspace/voxcpm2_test_{i}.wav"
        sf.write(out, wav, sr)
        print(f"   ✅ 时长 {dur:.2f}s | 合成 {synth:.2f}s | RTF {rtf:.3f} | {out}")
    except Exception as e:
        print(f"   ❌ {type(e).__name__}: {e}")
        import traceback; traceback.print_exc()

if total_rtf > 0:
    print(f"\n平均 RTF: {total_rtf/len(texts):.3f}")

In [ ]:
# 验证生成的音频文件 + 显存占用
import os, soundfile as sf, torch
test_dir = "/mnt/workspace"
print("=== 生成的音频文件 ===")
for f in sorted(os.listdir(test_dir)):
    if f.startswith("voxcpm2_test_") and f.endswith(".wav"):
        path = f"{test_dir}/{f}"
        wav, sr = sf.read(path)
        dur = len(wav) / sr
        sz = os.path.getsize(path) / 1e6
        print(f"{f}: {dur:.2f}s, {sr}Hz, {sz:.2f}MB")
if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() // 1024 // 1024
    total = torch.cuda.get_device_properties(0).total_memory // 1024 // 1024
    print(f"\n显存: {alloc}/{total} MB")
print("\n=== 测试完成, 请下载 .wav 试听 ===")

## 转入算力池 Worker 模式（可选）
若已确认单机推理正常，且 Secrets（REDIS_HOST / R2_*）已注入真值，
可一键转入 Worker 模式，监听 Redis 队列接收任务并回传音频：

```python
exec(open("modelscope_worker.py").read())
```

预期：
- 从 Redis `tts:tasks` 队列拉取任务 -> 合成 -> 上传 R2 -> `tts:results`

## 结果分析

1. **模型加载**: 检查模型是否正确加载到 GPU
2. **推理速度**: 观察 RTF (Real-Time Factor = 合成时间 / 音频时长)
3. **显存占用**: 观察 GPU 显存占用
4. **音频质量**: 听取生成的音频质量

## 预期指标 (VoxCPM2 V100/A100 FP16):
- RTF (实时率): ~0.03-0.08（V100/A100 略优于 T4）
- 显存占用: ~10-12 GB (FP16)
- 音频采样率: 24kHz
- 支持: 零样本音色克隆、语速控制、情感控制

## ⚠️ ModelScope 限制提醒
- 免费 Notebook 单卡 GPU，按时长配额，适合开发/烟测
- 无持久公网 IP/URL，重启需重新下载+加载模型
- 适合作为算力池的拉模式(pull)节点，与 kaggle-01 / paddle-01 对等
- 生产环境建议使用 Modal (A10G/V100) + 固定端点